In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
"""
================================================================================
QATAR AIRWAYS — A/B TEST ANALYSIS
Elevated Meal Service Trial | Apr–Jul 2024
Senior Data Specialist Portfolio — Product Development & Design Team
================================================================================

EXPERIMENT CONTEXT:
    Qatar Airways trialled an elevated economy and business class meal service
    (Treatment) against the existing menu (Control) across 4 long-haul routes
    over a 4-month period. The hypothesis: improving meal quality will lift
    overall passenger satisfaction and reduce complaint rates.

    Control   : Existing meal service (n=3,046)
    Treatment : Elevated menu — new supplier, upgraded presentation (n=2,954)
    Routes    : DOH-LHR, DOH-JFK, DOH-CDG, DOH-SIN
    Period    : April – July 2024

METRICS:
    Primary   : meal_satisfaction (1–5 rating)
    Secondary : meal_complaint (binary), meal_finish_rate (0–1),
                overall_sat_impact (downstream satisfaction effect)

OUTPUTS (saved to ./charts/):
    fig_ab_01_overview.png
    fig_ab_02_statistical_tests.png
    fig_ab_03_subgroup_analysis.png
    fig_ab_04_business_case.png
    fig_ab_05_executive_summary.png
================================================================================
"""

'\n================================================================================\nQATAR AIRWAYS — A/B TEST ANALYSIS\nElevated Meal Service Trial | Apr–Jul 2024\nSenior Data Specialist Portfolio — Product Development & Design Team\n================================================================================\n\nEXPERIMENT CONTEXT:\n    Qatar Airways trialled an elevated economy and business class meal service\n    (Treatment) against the existing menu (Control) across 4 long-haul routes\n    over a 4-month period. The hypothesis: improving meal quality will lift\n    overall passenger satisfaction and reduce complaint rates.\n\n    Control   : Existing meal service (n=3,046)\n    Treatment : Elevated menu — new supplier, upgraded presentation (n=2,954)\n    Routes    : DOH-LHR, DOH-JFK, DOH-CDG, DOH-SIN\n    Period    : April – July 2024\n\nMETRICS:\n    Primary   : meal_satisfaction (1–5 rating)\n    Secondary : meal_complaint (binary), meal_finish_rate (0–1),\n                ov

In [3]:


# ── IMPORTS ───────────────────────────────────────────────────────────────────
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import (
    ttest_ind, mannwhitneyu, chi2_contingency,
    norm, shapiro
)
import itertools

warnings.filterwarnings("ignore")
os.makedirs("charts", exist_ok=True)

In [4]:

# ── QATAR AIRWAYS BRAND PALETTE ───────────────────────────────────────────────
C = {
    "maroon":      "#8D1B3D",
    "maroon_light":"#B5466A",
    "steel_dark":  "#4A6278",
    "steel_mid":   "#6B8296",
    "grey_dark":   "#5A5A5A",
    "grey_light":  "#C8C9CA",
    "off_white":   "#F7F5F2",
    "positive":    "#2E7D5A",
    "negative":    "#C0392B",
    "warning":     "#C47B00",
    "control":     "#4A6278",   # steel dark for control
    "treatment":   "#8D1B3D",   # maroon for treatment
}




plt.rcParams.update({
    "figure.facecolor":   C["off_white"],
    "axes.facecolor":     C["off_white"],
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.edgecolor":     C["grey_light"],
    "axes.linewidth":     0.8,
    "axes.grid":          True,
    "grid.alpha":         0.2,
    "grid.linestyle":     "--",
    "grid.color":         C["grey_light"],
    "font.family":        "sans-serif",
    "font.size":          11,
    "axes.titlesize":     13,
    "axes.titleweight":   "bold",
    "axes.labelsize":     11,
    "xtick.labelsize":    10,
    "ytick.labelsize":    10,
    "legend.fontsize":    10,
    "legend.framealpha":  0.9,
    "legend.edgecolor":   C["grey_light"],
})

def save(fig, name):
    path = f"charts/{name}"
    fig.savefig(path, dpi=180, bbox_inches="tight", facecolor=C["off_white"])
    print(f"  ✓ Saved {path}")
    plt.close(fig)

def watermark(fig):
    fig.text(0.99, 0.01,
             "Qatar Airways | Meal Service A/B Test | PDD Portfolio",
             ha="right", va="bottom", fontsize=8,
             color=C["grey_dark"], alpha=0.5)

def sig_stars(p):
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "ns"

def cohens_d(a, b):
    """Effect size for two independent groups."""
    pooled_std = np.sqrt((np.std(a, ddof=1)**2 + np.std(b, ddof=1)**2) / 2)
    return (np.mean(a) - np.mean(b)) / pooled_std

def effect_label(d):
    d = abs(d)
    if d >= 0.8:  return "Large"
    if d >= 0.5:  return "Medium"
    if d >= 0.2:  return "Small"
    return "Negligible"

def add_sig_bracket(ax, x1, x2, y, p, h=0.05):
    """Draw a significance bracket between two bars."""
    stars = sig_stars(p)
    if stars == "ns":
        return
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.2, color=C["grey_dark"])
    ax.text((x1+x2)/2, y+h+0.01, stars, ha="center", va="bottom",
            fontsize=12, color=C["grey_dark"], fontweight="bold")


In [5]:

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
print("Loading A/B test data...")
df = pd.read_csv("/content/drive/MyDrive/Qatar_Job/Claude Synthetic Data/04_ab_test_meal.csv")
df["flight_date"] = pd.to_datetime(df["flight_date"])

ctrl = df[df["test_group"] == "Control"]
treat = df[df["test_group"] == "Treatment"]

print(f"  Control:   {len(ctrl):,} passengers")
print(f"  Treatment: {len(treat):,} passengers")
print()

Loading A/B test data...
  Control:   3,046 passengers
  Treatment: 2,954 passengers



In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   passenger_id        6000 non-null   object        
 1   flight_date         6000 non-null   datetime64[ns]
 2   month               6000 non-null   int64         
 3   route               6000 non-null   object        
 4   cabin_class         6000 non-null   object        
 5   loyalty_tier        6000 non-null   object        
 6   test_group          6000 non-null   object        
 7   meal_satisfaction   6000 non-null   float64       
 8   overall_sat_impact  6000 non-null   float64       
 9   meal_complaint      6000 non-null   int64         
 10  meal_finish_rate    6000 non-null   float64       
 11  meal_type           6000 non-null   object        
dtypes: datetime64[ns](1), float64(3), int64(2), object(6)
memory usage: 562.6+ KB


In [9]:


# ── STATISTICAL TESTS ─────────────────────────────────────────────────────────
print("Running statistical tests...")

# Primary metric — meal_satisfaction
t_stat, t_p    = ttest_ind(treat["meal_satisfaction"], ctrl["meal_satisfaction"])
u_stat, u_p    = mannwhitneyu(treat["meal_satisfaction"], ctrl["meal_satisfaction"],
                               alternative="greater")
d              = cohens_d(treat["meal_satisfaction"], ctrl["meal_satisfaction"])
treat_mean     = treat["meal_satisfaction"].mean()
ctrl_mean      = ctrl["meal_satisfaction"].mean()
lift_abs       = treat_mean - ctrl_mean
lift_pct       = lift_abs / ctrl_mean * 100

# Secondary — complaint rate (chi-squared)
complaint_ct   = pd.crosstab(df["test_group"], df["meal_complaint"])
chi2_c, p_c, _, _ = chi2_contingency(complaint_ct)
ctrl_complaint_rate  = ctrl["meal_complaint"].mean() * 100
treat_complaint_rate = treat["meal_complaint"].mean() * 100
complaint_reduction  = ctrl_complaint_rate - treat_complaint_rate

# Secondary — finish rate (t-test)
t_fr, p_fr     = ttest_ind(treat["meal_finish_rate"], ctrl["meal_finish_rate"])
d_fr           = cohens_d(treat["meal_finish_rate"], ctrl["meal_finish_rate"])

# Secondary — overall satisfaction impact
t_oi, p_oi     = ttest_ind(treat["overall_sat_impact"], ctrl["overall_sat_impact"])

# Normality check (Shapiro on sample for speed)
_, p_norm_ctrl  = shapiro(ctrl["meal_satisfaction"].sample(200, random_state=42))
_, p_norm_treat = shapiro(treat["meal_satisfaction"].sample(200, random_state=42))

# Confidence intervals (95%)
def ci_mean(data, confidence=0.95):
    n, m, se = len(data), np.mean(data), stats.sem(data)
    h = se * stats.t.ppf((1 + confidence) / 2, n - 1)
    return m - h, m + h

ctrl_ci  = ci_mean(ctrl["meal_satisfaction"])
treat_ci = ci_mean(treat["meal_satisfaction"])

print(f"  Primary — meal_satisfaction:")
print(f"    Control mean  : {ctrl_mean:.3f}  95% CI [{ctrl_ci[0]:.3f}, {ctrl_ci[1]:.3f}]")
print(f"    Treatment mean: {treat_mean:.3f}  95% CI [{treat_ci[0]:.3f}, {treat_ci[1]:.3f}]")
print(f"    Absolute lift : +{lift_abs:.3f} pts  ({lift_pct:.1f}%)")
print(f"    t-test        : t={t_stat:.3f}, p={t_p:.6f} {sig_stars(t_p)}")
print(f"    Mann-Whitney  : U={u_stat:.0f}, p={u_p:.6f} {sig_stars(u_p)}")
print(f"    Cohen's d     : {d:.3f} ({effect_label(d)} effect)")
print()
print(f"  Secondary — complaint rate:")
print(f"    Control  : {ctrl_complaint_rate:.1f}%")
print(f"    Treatment: {treat_complaint_rate:.1f}%")
print(f"    Reduction: -{complaint_reduction:.1f}pp")
print(f"    Chi²     : χ²={chi2_c:.3f}, p={p_c:.6f} {sig_stars(p_c)}")
print()
print(f"  Secondary — finish rate:")
print(f"    Control  : {ctrl['meal_finish_rate'].mean():.3f}")
print(f"    Treatment: {treat['meal_finish_rate'].mean():.3f}")
print(f"    t-test   : t={t_fr:.3f}, p={p_fr:.6f} {sig_stars(p_fr)}")
print(f"    Cohen's d: {d_fr:.3f} ({effect_label(d_fr)} effect)")
print()

# Subgroup — cabin class
print("  Subgroup analysis (cabin):")
cabin_results = []
for cabin in ["Economy", "Business", "First"]:
    c_sub = ctrl[ctrl["cabin_class"] == cabin]["meal_satisfaction"]
    t_sub = treat[treat["cabin_class"] == cabin]["meal_satisfaction"]
    t_s, p_s = ttest_ind(t_sub, c_sub)
    d_s = cohens_d(t_sub, c_sub)
    lift_s = t_sub.mean() - c_sub.mean()
    cabin_results.append({
        "cabin": cabin, "ctrl_mean": c_sub.mean(), "treat_mean": t_sub.mean(),
        "lift": lift_s, "p": p_s, "d": d_s,
        "ctrl_n": len(c_sub), "treat_n": len(t_sub),
    })
    print(f"    {cabin:<10}: lift={lift_s:+.3f}  p={p_s:.4f} {sig_stars(p_s)}  d={d_s:.3f} ({effect_label(d_s)})")

cabin_df = pd.DataFrame(cabin_results)

# Subgroup — route
print("  Subgroup analysis (route):")
route_results = []
for route in df["route"].unique():
    c_sub = ctrl[ctrl["route"] == route]["meal_satisfaction"]
    t_sub = treat[treat["route"] == route]["meal_satisfaction"]
    t_s, p_s = ttest_ind(t_sub, c_sub)
    lift_s = t_sub.mean() - c_sub.mean()
    route_results.append({
        "route": route, "ctrl_mean": c_sub.mean(), "treat_mean": t_sub.mean(),
        "lift": lift_s, "p": p_s, "ctrl_n": len(c_sub), "treat_n": len(t_sub),
    })
    print(f"    {route}: lift={lift_s:+.3f}  p={p_s:.4f} {sig_stars(p_s)}")

route_df = pd.DataFrame(route_results)

# Subgroup — loyalty tier
loyalty_results = []
for tier in ["Blue", "Silver", "Gold", "Platinum"]:
    c_sub = ctrl[ctrl["loyalty_tier"] == tier]["meal_satisfaction"]
    t_sub = treat[treat["loyalty_tier"] == tier]["meal_satisfaction"]
    t_s, p_s = ttest_ind(t_sub, c_sub)
    lift_s = t_sub.mean() - c_sub.mean()
    loyalty_results.append({
        "tier": tier, "ctrl_mean": c_sub.mean(), "treat_mean": t_sub.mean(),
        "lift": lift_s, "p": p_s,
    })
loyalty_df = pd.DataFrame(loyalty_results)

# Monthly trend
monthly = df.groupby(["month", "test_group"])["meal_satisfaction"].mean().unstack()
month_labels = {4: "Apr", 5: "May", 6: "Jun", 7: "Jul"}

print()

# ── BUSINESS CASE PROJECTIONS ─────────────────────────────────────────────────
annual_pax_economy  = 25_000_000   # illustrative QA economy passengers p.a.
annual_pax_business = 5_000_000
complaint_cost_usd  = 85            # avg cost to handle one in-flight complaint
economy_meal_cost_uplift = 4.50     # estimated cost uplift per meal (Treatment)
business_meal_cost_uplift = 12.00

complaints_avoided_economy  = (ctrl_complaint_rate/100 - treat_complaint_rate/100) * annual_pax_economy
complaints_avoided_business = (
    (cabin_df[cabin_df["cabin"]=="Business"]["ctrl_mean"].values[0] * 0.08) -
    (cabin_df[cabin_df["cabin"]=="Business"]["treat_mean"].values[0] * 0.04)
) * annual_pax_business

saving_complaints = complaints_avoided_economy * complaint_cost_usd
cost_of_treatment_economy  = annual_pax_economy  * economy_meal_cost_uplift
cost_of_treatment_business = annual_pax_business * business_meal_cost_uplift
net_saving = saving_complaints - cost_of_treatment_economy - cost_of_treatment_business

nps_uplift_proxy = lift_abs * 8   # rough conversion: 1pt meal sat ≈ 8 NPS pts

Running statistical tests...
  Primary — meal_satisfaction:
    Control mean  : 3.607  95% CI [3.589, 3.625]
    Treatment mean: 3.906  95% CI [3.889, 3.923]
    Absolute lift : +0.299 pts  (8.3%)
    t-test        : t=23.547, p=0.000000 ***
    Mann-Whitney  : U=6028401, p=0.000000 ***
    Cohen's d     : 0.609 (Medium effect)

  Secondary — complaint rate:
    Control  : 8.0%
    Treatment: 4.3%
    Reduction: -3.7pp
    Chi²     : χ²=34.969, p=0.000000 ***

  Secondary — finish rate:
    Control  : 0.777
    Treatment: 0.876
    t-test   : t=40.869, p=0.000000 ***
    Cohen's d: 1.056 (Large effect)

  Subgroup analysis (cabin):
    Economy   : lift=+0.341  p=0.0000 ***  d=0.850 (Large)
    Business  : lift=+0.215  p=0.0000 ***  d=0.554 (Medium)
    First     : lift=+0.076  p=0.0474 *  d=0.216 (Small)
  Subgroup analysis (route):
    DOH-LHR: lift=+0.298  p=0.0000 ***
    DOH-SIN: lift=+0.300  p=0.0000 ***
    DOH-JFK: lift=+0.307  p=0.0000 ***
    DOH-CDG: lift=+0.292  p=0.0000 ***

In [10]:

# ════════════════════════════════════════════════════════════════════════════
# FIG AB-01 — EXPERIMENT OVERVIEW
# ════════════════════════════════════════════════════════════════════════════
print("FIG AB-01 — Experiment overview...")

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle(
    "Meal Service A/B Test — Experiment Overview\n"
    "Control (n=3,046) vs Treatment — Elevated Menu (n=2,954)",
    fontsize=16, fontweight="bold", y=1.01
)

# 1a — Group balance check (sample sizes by cabin)
ax = axes[0, 0]
balance = df.groupby(["cabin_class", "test_group"]).size().unstack()
x = np.arange(len(balance))
w = 0.35
ax.bar(x - w/2, balance["Control"],   w, color=C["control"],   label="Control",   edgecolor="none")
ax.bar(x + w/2, balance["Treatment"], w, color=C["treatment"], label="Treatment", edgecolor="none")
ax.set_xticks(x); ax.set_xticklabels(balance.index)
ax.set_title("Sample Balance by Cabin Class", fontweight="bold")
ax.set_ylabel("Number of Passengers")
ax.legend()

# 1b — Route distribution balance
ax = axes[0, 1]
route_bal = df.groupby(["route", "test_group"]).size().unstack()
x = np.arange(len(route_bal))
ax.bar(x - w/2, route_bal["Control"],   w, color=C["control"],   label="Control",   edgecolor="none")
ax.bar(x + w/2, route_bal["Treatment"], w, color=C["treatment"], label="Treatment", edgecolor="none")
ax.set_xticks(x); ax.set_xticklabels(route_bal.index, rotation=20, ha="right")
ax.set_title("Sample Balance by Route", fontweight="bold")
ax.set_ylabel("Number of Passengers")
ax.legend()

# 1c — Loyalty tier balance
ax = axes[0, 2]
loyalty_bal = df.groupby(["loyalty_tier", "test_group"]).size().unstack()
loyalty_bal = loyalty_bal.reindex(["Blue", "Silver", "Gold", "Platinum"])
x = np.arange(len(loyalty_bal))
ax.bar(x - w/2, loyalty_bal["Control"],   w, color=C["control"],   label="Control",   edgecolor="none")
ax.bar(x + w/2, loyalty_bal["Treatment"], w, color=C["treatment"], label="Treatment", edgecolor="none")
ax.set_xticks(x); ax.set_xticklabels(loyalty_bal.index)
ax.set_title("Sample Balance by Loyalty Tier", fontweight="bold")
ax.set_ylabel("Number of Passengers"); ax.legend()

# 1d — Primary metric distribution (KDE)
ax = axes[1, 0]
for grp, col, label in [("Control", C["control"], "Control"),
                          ("Treatment", C["treatment"], "Treatment")]:
    data = df[df["test_group"] == grp]["meal_satisfaction"]
    data.plot.kde(ax=ax, color=col, linewidth=2.5, label=f"{label} (μ={data.mean():.2f})")
    ax.axvline(data.mean(), color=col, linestyle="--", linewidth=1.5, alpha=0.7)
ax.set_title("Meal Satisfaction Distribution (KDE)", fontweight="bold")
ax.set_xlabel("Meal Satisfaction (1–5)"); ax.set_ylabel("Density")
ax.legend(); ax.set_xlim(1, 5)

# 1e — Meal type mix (same across groups?)
ax = axes[1, 1]
meal_mix = df.groupby(["meal_type", "test_group"]).size().unstack()
meal_mix_pct = meal_mix.div(meal_mix.sum(axis=0), axis=1) * 100
x = np.arange(len(meal_mix_pct))
ax.bar(x - w/2, meal_mix_pct["Control"],   w, color=C["control"],   label="Control",   edgecolor="none")
ax.bar(x + w/2, meal_mix_pct["Treatment"], w, color=C["treatment"], label="Treatment", edgecolor="none")
ax.set_xticks(x); ax.set_xticklabels(meal_mix_pct.index, rotation=15, ha="right")
ax.set_title("Meal Type Mix — Group Comparison", fontweight="bold")
ax.set_ylabel("% of Passengers"); ax.legend()

# 1f — Monthly volume by group
ax = axes[1, 2]
month_vol = df.groupby(["month", "test_group"]).size().unstack()
x = np.arange(len(month_vol))
ax.bar(x - w/2, month_vol["Control"],   w, color=C["control"],   label="Control",   edgecolor="none")
ax.bar(x + w/2, month_vol["Treatment"], w, color=C["treatment"], label="Treatment", edgecolor="none")
ax.set_xticks(x)
ax.set_xticklabels([month_labels[m] for m in month_vol.index])
ax.set_title("Monthly Sample Volume", fontweight="bold")
ax.set_ylabel("Number of Passengers"); ax.legend()

plt.tight_layout()
watermark(fig)
save(fig, "fig_ab_01_overview.png")



FIG AB-01 — Experiment overview...
  ✓ Saved charts/fig_ab_01_overview.png


In [11]:

# ════════════════════════════════════════════════════════════════════════════
# FIG AB-02 — STATISTICAL TESTS
# ════════════════════════════════════════════════════════════════════════════
print("FIG AB-02 — Statistical tests...")

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle(
    "A/B Test — Statistical Analysis\n"
    "Primary & Secondary Metric Significance Testing",
    fontsize=16, fontweight="bold", y=1.01
)

# 2a — Primary metric bar with CI
ax = axes[0, 0]
means  = [ctrl_mean,   treat_mean]
ci_lo  = [ctrl_mean  - ctrl_ci[0],  treat_mean  - treat_ci[0]]
ci_hi  = [ctrl_ci[1]  - ctrl_mean,  treat_ci[1] - treat_mean]
bars   = ax.bar([0, 1], means, color=[C["control"], C["treatment"]],
                edgecolor="none", width=0.5)
ax.errorbar([0, 1], means, yerr=[ci_lo, ci_hi],
            fmt="none", color=C["grey_dark"], capsize=8, linewidth=2)
ax.set_xticks([0, 1]); ax.set_xticklabels(["Control", "Treatment"])
ax.set_ylim(3.0, 4.5)
ax.set_title("Meal Satisfaction — Mean ± 95% CI", fontweight="bold")
ax.set_ylabel("Avg Meal Satisfaction (1–5)")
add_sig_bracket(ax, 0, 1, max(means) + 0.15, t_p, h=0.06)
ax.text(0.5, 3.1, f"Δ = +{lift_abs:.3f} pts ({lift_pct:.1f}%)\n"
        f"t={t_stat:.2f}, p={t_p:.4f} {sig_stars(t_p)}\n"
        f"Cohen's d = {d:.3f} ({effect_label(d)})",
        ha="center", va="bottom", fontsize=9,
        color=C["maroon"], fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                  edgecolor=C["maroon"], alpha=0.85))

# 2b — Complaint rate
ax = axes[0, 1]
complaint_rates = [ctrl_complaint_rate, treat_complaint_rate]
bar_cols = [C["control"], C["treatment"]]
bars = ax.bar([0, 1], complaint_rates, color=bar_cols,
              edgecolor="none", width=0.5)
for bar, val in zip(bars, complaint_rates):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.3,
            f"{val:.1f}%", ha="center", va="bottom",
            fontsize=11, fontweight="bold")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Control", "Treatment"])
ax.set_title("Complaint Rate (% of Passengers)", fontweight="bold")
ax.set_ylabel("Complaint Rate (%)")
ax.set_ylim(0, 14)
add_sig_bracket(ax, 0, 1, max(complaint_rates) + 1.5, p_c, h=0.5)
ax.text(0.5, 1.5, f"−{complaint_reduction:.1f}pp reduction\n"
        f"χ²={chi2_c:.2f}, p={p_c:.4f} {sig_stars(p_c)}",
        ha="center", fontsize=9, color=C["positive"], fontweight="bold")

# 2c — Finish rate
ax = axes[0, 2]
finish_rates = [ctrl["meal_finish_rate"].mean()*100,
                treat["meal_finish_rate"].mean()*100]
bars = ax.bar([0, 1], finish_rates, color=[C["control"], C["treatment"]],
              edgecolor="none", width=0.5)
for bar, val in zip(bars, finish_rates):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.3,
            f"{val:.1f}%", ha="center", va="bottom",
            fontsize=11, fontweight="bold")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Control", "Treatment"])
ax.set_title("Meal Finish Rate (%)", fontweight="bold")
ax.set_ylabel("Avg Finish Rate (%)")
ax.set_ylim(60, 100)
add_sig_bracket(ax, 0, 1, max(finish_rates) + 1.5, p_fr, h=0.5)
ax.text(0.5, 62,
        f"t={t_fr:.2f}, p={p_fr:.4f} {sig_stars(p_fr)}\n"
        f"d={d_fr:.3f} ({effect_label(d_fr)})",
        ha="center", fontsize=9, color=C["positive"], fontweight="bold")

# 2d — Violin plot: full distribution comparison
ax = axes[1, 0]
parts = ax.violinplot(
    [ctrl["meal_satisfaction"], treat["meal_satisfaction"]],
    positions=[0, 1], showmedians=True, showextrema=True
)
for pc, col in zip(parts["bodies"], [C["control"], C["treatment"]]):
    pc.set_facecolor(col); pc.set_alpha(0.7)
parts["cmedians"].set_color(C["off_white"]); parts["cmedians"].set_linewidth(2.5)
ax.set_xticks([0, 1]); ax.set_xticklabels(["Control", "Treatment"])
ax.set_title("Score Distribution — Violin Plot", fontweight="bold")
ax.set_ylabel("Meal Satisfaction (1–5)")

# 2e — Cumulative distribution comparison
ax = axes[1, 1]
for grp, col, label in [("Control", C["control"], "Control"),
                          ("Treatment", C["treatment"], "Treatment")]:
    data = np.sort(df[df["test_group"]==grp]["meal_satisfaction"])
    cdf  = np.arange(1, len(data)+1) / len(data)
    ax.plot(data, cdf, color=col, linewidth=2.5, label=label)
ax.set_title("Cumulative Distribution Function", fontweight="bold")
ax.set_xlabel("Meal Satisfaction Score (1–5)")
ax.set_ylabel("Cumulative Proportion")
ax.legend(); ax.set_xlim(1, 5)
ax.axhline(0.5, color=C["grey_dark"], linestyle=":", linewidth=1, alpha=0.6)

# 2f — Statistical test summary table
ax = axes[1, 2]
ax.axis("off")
table_data = [
    ["Metric",           "Statistic",          "p-value", "Sig.", "Effect"],
    ["Meal Satisfaction",f"t = {t_stat:.3f}",  f"{t_p:.5f}", sig_stars(t_p), f"d={d:.3f} ({effect_label(d)})"],
    ["Meal Satisfaction",f"U = {u_stat:.0f}",  f"{u_p:.5f}", sig_stars(u_p), "Non-param confirmed"],
    ["Complaint Rate",   f"χ² = {chi2_c:.3f}", f"{p_c:.5f}", sig_stars(p_c), f"−{complaint_reduction:.1f}pp"],
    ["Finish Rate",      f"t = {t_fr:.3f}",    f"{p_fr:.5f}", sig_stars(p_fr), f"d={d_fr:.3f} ({effect_label(d_fr)})"],
    ["Overall Sat.",     f"t = {t_oi:.3f}",    f"{p_oi:.5f}", sig_stars(p_oi), "Downstream effect"],
]
table = ax.table(
    cellText=table_data[1:],
    colLabels=table_data[0],
    loc="center", cellLoc="center"
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.8)
for (row, col), cell in table.get_celld().items():
    cell.set_facecolor(C["maroon"] if row == 0 else
                       C["off_white"] if row % 2 == 0 else "white")
    cell.set_text_props(color="white" if row == 0 else C["grey_dark"],
                        fontweight="bold" if row == 0 else "normal")
    cell.set_edgecolor(C["grey_light"])
ax.set_title("Statistical Test Summary", fontweight="bold", y=0.92)

plt.tight_layout()
watermark(fig)
save(fig, "fig_ab_02_statistical_tests.png")



FIG AB-02 — Statistical tests...
  ✓ Saved charts/fig_ab_02_statistical_tests.png


In [12]:

# ════════════════════════════════════════════════════════════════════════════
# FIG AB-03 — SUBGROUP ANALYSIS
# ════════════════════════════════════════════════════════════════════════════
print("FIG AB-03 — Subgroup analysis...")

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle(
    "Subgroup Analysis — Does Treatment Effect Vary?\n"
    "Cabin Class · Route · Loyalty Tier · Month",
    fontsize=16, fontweight="bold", y=1.01
)

# 3a — Treatment lift by cabin (forest plot style)
ax = axes[0, 0]
cabins    = cabin_df["cabin"]
lifts     = cabin_df["lift"]
p_vals    = cabin_df["p"]
colours   = [C["positive"] if p < 0.05 else C["warning"] for p in p_vals]
bars      = ax.barh(cabins, lifts, color=colours, edgecolor="none", height=0.5)
ax.axvline(0, color=C["grey_dark"], linewidth=1)
ax.axvline(lift_abs, color=C["maroon"], linestyle="--",
           linewidth=1.5, label=f"Overall lift ({lift_abs:.3f})")
for i, (lift, p) in enumerate(zip(lifts, p_vals)):
    ax.text(lift + 0.005, i, f"{lift:+.3f} {sig_stars(p)}",
            va="center", fontsize=10, fontweight="bold",
            color=C["positive"] if p < 0.05 else C["warning"])
ax.set_title("Treatment Lift by Cabin Class", fontweight="bold")
ax.set_xlabel("Lift in Meal Satisfaction (pts)")
ax.legend(fontsize=9)

# 3b — Cabin group means side-by-side
ax = axes[0, 1]
x = np.arange(len(cabin_df))
w = 0.35
ax.bar(x - w/2, cabin_df["ctrl_mean"],  w, color=C["control"],
       label="Control",   edgecolor="none")
ax.bar(x + w/2, cabin_df["treat_mean"], w, color=C["treatment"],
       label="Treatment", edgecolor="none")
ax.set_xticks(x); ax.set_xticklabels(cabin_df["cabin"])
ax.set_title("Meal Satisfaction by Cabin & Group", fontweight="bold")
ax.set_ylabel("Avg Satisfaction (1–5)")
ax.set_ylim(3.0, 5.0); ax.legend()
for i, row in cabin_df.iterrows():
    ax.text(i + w/2, row["treat_mean"] + 0.04,
            sig_stars(row["p"]), ha="center", fontsize=11,
            color=C["maroon"], fontweight="bold")

# 3c — Lift by route
ax = axes[0, 2]
route_df_s = route_df.sort_values("lift", ascending=True)
r_colours  = [C["positive"] if p < 0.05 else C["warning"]
               for p in route_df_s["p"]]
ax.barh(route_df_s["route"], route_df_s["lift"],
        color=r_colours, edgecolor="none", height=0.5)
ax.axvline(0, color=C["grey_dark"], linewidth=1)
ax.axvline(lift_abs, color=C["maroon"], linestyle="--",
           linewidth=1.5, label=f"Overall lift")
for i, (_, row) in enumerate(route_df_s.iterrows()):
    ax.text(row["lift"] + 0.003, i,
            f"{row['lift']:+.3f} {sig_stars(row['p'])}",
            va="center", fontsize=10)
ax.set_title("Treatment Lift by Route", fontweight="bold")
ax.set_xlabel("Lift in Meal Satisfaction (pts)")
ax.legend(fontsize=9)

# 3d — Lift by loyalty tier
ax = axes[1, 0]
loyalty_df_s = loyalty_df.set_index("tier").reindex(
    ["Blue","Silver","Gold","Platinum"])
tier_colours = [C["positive"] if p < 0.05 else C["warning"]
                for p in loyalty_df_s["p"]]
bars = ax.bar(loyalty_df_s.index, loyalty_df_s["lift"],
              color=tier_colours, edgecolor="none", width=0.55)
ax.axhline(0, color=C["grey_dark"], linewidth=1)
ax.axhline(lift_abs, color=C["maroon"], linestyle="--",
           linewidth=1.5, label=f"Overall lift")
for bar, (_, row) in zip(bars, loyalty_df_s.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2,
            row["lift"] + 0.005, f"{row['lift']:+.3f}",
            ha="center", va="bottom", fontsize=9)
ax.set_title("Treatment Lift by Loyalty Tier", fontweight="bold")
ax.set_ylabel("Lift in Meal Satisfaction (pts)")
ax.legend(fontsize=9)

# 3e — Monthly trend of satisfaction score
ax = axes[1, 1]
if "Control" in monthly.columns and "Treatment" in monthly.columns:
    ax.plot(monthly.index, monthly["Control"],
            "o-", color=C["control"], linewidth=2.2,
            markersize=7, label="Control")
    ax.plot(monthly.index, monthly["Treatment"],
            "s-", color=C["treatment"], linewidth=2.2,
            markersize=7, label="Treatment")
    ax.fill_between(monthly.index,
                    monthly["Control"], monthly["Treatment"],
                    alpha=0.12, color=C["treatment"], label="Gap")
ax.set_xticks(list(month_labels.keys()))
ax.set_xticklabels(list(month_labels.values()))
ax.set_title("Satisfaction Trend — Control vs Treatment", fontweight="bold")
ax.set_ylabel("Avg Meal Satisfaction (1–5)")
ax.legend()

# 3f — Complaint rate heatmap (cabin × group)
ax = axes[1, 2]
complaint_heat = df.groupby(["cabin_class","test_group"])["meal_complaint"].mean() * 100
complaint_heat = complaint_heat.unstack()
from matplotlib.colors import LinearSegmentedColormap
RED_CMAP = LinearSegmentedColormap.from_list(
    "red_cmap", [C["off_white"], C["warning"], C["negative"]])
sns.heatmap(complaint_heat, ax=ax, cmap=RED_CMAP,
            annot=True, fmt=".1f", annot_kws={"size": 11},
            linewidths=0.5, cbar_kws={"label":"Complaint Rate (%)", "shrink":0.8})
ax.set_title("Complaint Rate (%) — Cabin × Group", fontweight="bold")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

plt.tight_layout()
watermark(fig)
save(fig, "fig_ab_03_subgroup_analysis.png")



FIG AB-03 — Subgroup analysis...
  ✓ Saved charts/fig_ab_03_subgroup_analysis.png


In [13]:

# ════════════════════════════════════════════════════════════════════════════
# FIG AB-04 — BUSINESS CASE
# ════════════════════════════════════════════════════════════════════════════
print("FIG AB-04 — Business case...")

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle(
    "Business Case — Should Qatar Airways Roll Out the Elevated Meal Service?\n"
    "Projections Based on Trial Results",
    fontsize=16, fontweight="bold", y=1.01
)

# 4a — Complaints avoided vs cost (waterfall-style)
ax = axes[0, 0]
categories = [
    "Complaint\nsavings\n(Economy)",
    "Implementation\ncost (Economy)",
    "Implementation\ncost (Business)",
    "Net\nposition",
]
values = [
    complaints_avoided_economy * complaint_cost_usd / 1e6,
    -cost_of_treatment_economy / 1e6,
    -cost_of_treatment_business / 1e6,
    net_saving / 1e6,
]
bar_cols = [C["positive"] if v >= 0 else C["negative"] for v in values]
bars = ax.bar(categories, values, color=bar_cols, edgecolor="none", width=0.55)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2,
            val + (0.5 if val >= 0 else -1.5),
            f"${val:.1f}M", ha="center", va="bottom",
            fontsize=10, fontweight="bold",
            color=C["positive"] if val >= 0 else C["negative"])
ax.axhline(0, color=C["grey_dark"], linewidth=1)
ax.set_title("Estimated Annual Financial Impact (USD)", fontweight="bold")
ax.set_ylabel("USD Millions")

# 4b — Complaints avoided breakdown
ax = axes[0, 1]
trial_ctrl  = ctrl_complaint_rate
trial_treat = treat_complaint_rate
proj_cabins = ["Economy\n(Projected)", "Business\n(Projected)", "First\n(Projected)"]
proj_ctrl   = [cabin_df[cabin_df["cabin"]==c]["ctrl_mean"].values[0] * 8
               for c in ["Economy", "Business", "First"]]
proj_treat  = [cabin_df[cabin_df["cabin"]==c]["treat_mean"].values[0] * 8
               for c in ["Economy", "Business", "First"]]
x = np.arange(len(proj_cabins))
w = 0.3
ax.bar(x - w/2, [cabin_df[cabin_df["cabin"]==c]["ctrl_mean"].values[0]
                  for c in ["Economy","Business","First"]],
       w, color=C["control"],   label="Control",   edgecolor="none")
ax.bar(x + w/2, [cabin_df[cabin_df["cabin"]==c]["treat_mean"].values[0]
                  for c in ["Economy","Business","First"]],
       w, color=C["treatment"], label="Treatment", edgecolor="none")
ax.set_xticks(x); ax.set_xticklabels(proj_cabins)
ax.set_title("Projected Mean Satisfaction by Cabin", fontweight="bold")
ax.set_ylabel("Avg Meal Satisfaction (1–5)")
ax.set_ylim(3.0, 5.0); ax.legend()

# 4c — NPS proxy impact
ax = axes[1, 0]
nps_scenarios = {
    "Economy only\n(70% pax)":   lift_abs * 0.7 * 8,
    "Economy +\nBusiness":       (cabin_df[cabin_df["cabin"]=="Economy"]["lift"].values[0]*0.7 +
                                   cabin_df[cabin_df["cabin"]=="Business"]["lift"].values[0]*0.24) * 8,
    "Full fleet\nrollout":       lift_abs * 8,
}
bar_cols = [C["steel_mid"], C["steel_dark"], C["maroon"]]
bars = ax.bar(list(nps_scenarios.keys()), list(nps_scenarios.values()),
              color=bar_cols, edgecolor="none", width=0.5)
for bar, val in zip(bars, nps_scenarios.values()):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.05,
            f"+{val:.1f} pts", ha="center", va="bottom",
            fontsize=10, fontweight="bold")
ax.set_title("Projected NPS Proxy Improvement — Rollout Scenarios", fontweight="bold")
ax.set_ylabel("Estimated NPS Point Improvement")
ax.set_ylim(0, max(nps_scenarios.values()) * 1.3)

# 4d — Decision framework text panel
ax = axes[1, 1]
ax.axis("off")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.add_patch(plt.Rectangle((0,0), 1, 1, facecolor="white",
                             edgecolor=C["maroon"], linewidth=2))

lines = [
    (0.5, 0.94, "RECOMMENDATION", 13, C["maroon"], "bold"),
    (0.5, 0.86, "ROLL OUT TREATMENT MENU", 15, C["maroon"], "bold"),
    (0.5, 0.79, "Confidence: HIGH  |  Priority: IMMEDIATE", 10, C["steel_dark"], "normal"),
    (0.08, 0.70, "Evidence:", 10, C["grey_dark"], "bold"),
    (0.08, 0.63, f"✓  +{lift_abs:.2f}pt meal satisfaction lift  ({sig_stars(t_p)})", 10, C["positive"], "normal"),
    (0.08, 0.56, f"✓  Complaint rate halved  ({ctrl_complaint_rate:.0f}% → {treat_complaint_rate:.0f}%)", 10, C["positive"], "normal"),
    (0.08, 0.49, f"✓  Finish rate +{(treat['meal_finish_rate'].mean()-ctrl['meal_finish_rate'].mean())*100:.0f}pp — behavioural confirmation", 10, C["positive"], "normal"),
    (0.08, 0.42, f"✓  Effect consistent across all 4 routes", 10, C["positive"], "normal"),
    (0.08, 0.35, f"✓  Statistically significant in Economy & Business", 10, C["positive"], "normal"),
    (0.08, 0.26, "Caveat:", 10, C["grey_dark"], "bold"),
    (0.08, 0.19, "~  First class lift negligible — already near ceiling", 10, C["warning"], "normal"),
    (0.08, 0.12, "~  Trial period 4 months — recommend 12-month validation", 10, C["warning"], "normal"),
    (0.08, 0.05, "~  Cost uplift requires CFO sign-off on per-meal budget", 10, C["warning"], "normal"),
]
for x, y, text, size, col, weight in lines:
    ax.text(x, y, text, transform=ax.transAxes,
            fontsize=size, color=col, fontweight=weight,
            ha="center" if x == 0.5 else "left", va="center")

plt.tight_layout()
watermark(fig)
save(fig, "fig_ab_04_business_case.png")



FIG AB-04 — Business case...
  ✓ Saved charts/fig_ab_04_business_case.png


In [14]:

# ════════════════════════════════════════════════════════════════════════════
# FIG AB-05 — EXECUTIVE SUMMARY
# ════════════════════════════════════════════════════════════════════════════
print("FIG AB-05 — Executive summary...")

fig = plt.figure(figsize=(20, 13))
fig.suptitle(
    "Meal Service A/B Test — Executive Summary Dashboard\n"
    "Qatar Airways Product Development & Design | Senior Data Specialist Portfolio",
    fontsize=16, fontweight="bold", y=1.01, color=C["maroon"]
)
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.55, wspace=0.4)

# KPI tiles
kpis = [
    (f"+{lift_abs:.2f}",  "Meal Satisfaction Lift",  "pts above control",     C["maroon"]),
    (f"{sig_stars(t_p)}", "Statistical Significance", f"p = {t_p:.5f}",        C["steel_dark"]),
    (f"−{complaint_reduction:.0f}pp", "Complaint Rate Reduction",
                                         f"{ctrl_complaint_rate:.0f}% → {treat_complaint_rate:.0f}%", C["positive"]),
    (f"{effect_label(d)}", "Effect Size",             f"Cohen's d = {d:.3f}",   C["steel_mid"]),
]
for i, (val, label, sub, col) in enumerate(kpis):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor(col)
    ax.text(0.5, 0.62, val,   ha="center", va="center", fontsize=26,
            fontweight="bold", color="white", transform=ax.transAxes)
    ax.text(0.5, 0.30, label, ha="center", va="center", fontsize=11,
            fontweight="bold", color="white", transform=ax.transAxes)
    ax.text(0.5, 0.10, sub,   ha="center", va="center", fontsize=9,
            color="white", alpha=0.85, transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

# Main KDE chart
ax = fig.add_subplot(gs[1, :2])
for grp, col, label in [("Control", C["control"], f"Control  μ={ctrl_mean:.3f}"),
                          ("Treatment", C["treatment"], f"Treatment  μ={treat_mean:.3f}")]:
    data = df[df["test_group"]==grp]["meal_satisfaction"]
    data.plot.kde(ax=ax, color=col, linewidth=2.5, label=label)
    ax.axvline(data.mean(), color=col, linestyle="--", linewidth=1.5, alpha=0.8)
ax.fill_betweenx([0, 3], ctrl_mean, treat_mean,
                 alpha=0.08, color=C["treatment"])
ax.set_title("Meal Satisfaction Distribution — Control vs Treatment", fontweight="bold")
ax.set_xlabel("Meal Satisfaction (1–5)"); ax.set_ylabel("Density")
ax.legend(); ax.set_xlim(1.5, 5.5)

# Cabin subgroup lifts
ax = fig.add_subplot(gs[1, 2])
colours_cabin = [C["positive"] if p < 0.05 else C["warning"]
                 for p in cabin_df["p"]]
ax.barh(cabin_df["cabin"], cabin_df["lift"],
        color=colours_cabin, edgecolor="none", height=0.5)
ax.axvline(0, color=C["grey_dark"], linewidth=1)
for i, (_, row) in enumerate(cabin_df.iterrows()):
    ax.text(row["lift"] + 0.003, i,
            f"{row['lift']:+.3f} {sig_stars(row['p'])}",
            va="center", fontsize=9, fontweight="bold")
ax.set_title("Lift by Cabin", fontweight="bold")
ax.set_xlabel("Satisfaction Lift (pts)")

# Monthly trend
ax = fig.add_subplot(gs[1, 3])
if "Control" in monthly.columns and "Treatment" in monthly.columns:
    ax.plot(monthly.index, monthly["Control"],
            "o-", color=C["control"], linewidth=2, markersize=6, label="Control")
    ax.plot(monthly.index, monthly["Treatment"],
            "s-", color=C["treatment"], linewidth=2, markersize=6, label="Treatment")
ax.set_xticks(list(month_labels.keys()))
ax.set_xticklabels(list(month_labels.values()))
ax.set_title("Monthly Trend", fontweight="bold")
ax.set_ylabel("Avg Meal Satisfaction")
ax.legend(fontsize=9)

# Finding cards (bottom row)
findings = [
    (C["positive"],  "01 — Primary Metric: SIGNIFICANT",
     f"Treatment lifted meal satisfaction by +{lift_abs:.2f}pts (t-test p={t_p:.4f}, "
     f"Cohen's d={d:.2f}). Effect confirmed by non-parametric Mann-Whitney test. "
     f"Result consistent across all 4 test routes."),
    (C["maroon"],    "02 — Complaint Rate Halved",
     f"In-flight complaints fell from {ctrl_complaint_rate:.0f}% to {treat_complaint_rate:.0f}% "
     f"(χ²={chi2_c:.2f}, p={p_c:.4f}). Behavioural metric (finish rate +{(treat['meal_finish_rate'].mean()-ctrl['meal_finish_rate'].mean())*100:.0f}pp) "
     f"independently confirms passenger preference — not just stated satisfaction."),
    (C["steel_dark"],"03 — Economy Cabin: Highest ROI",
     f"Economy shows largest lift (+{cabin_df[cabin_df['cabin']=='Economy']['lift'].values[0]:.2f}pts, "
     f"{sig_stars(cabin_df[cabin_df['cabin']=='Economy']['p'].values[0])}). "
     f"First Class lift negligible — already at satisfaction ceiling. "
     f"Prioritise Economy rollout for maximum impact per dollar spent."),
    (C["warning"],   "04 — Caveat: Validate Over 12 Months",
     "Trial ran Apr–Jul 2024 only. Summer travel behaviours may inflate results. "
     "Recommend full 12-month validation before permanent menu change. "
     "Monitor cost-per-meal uplift vs complaint-handling cost savings quarterly."),
]
for i, (col, title, body) in enumerate(findings):
    ax = fig.add_subplot(gs[2, i])
    ax.set_facecolor("white")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xticks([]); ax.set_yticks([])
    ax.spines["left"].set_color(col); ax.spines["left"].set_linewidth(4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["bottom"].set_visible(False)
    ax.text(0.06, 0.88, title, fontsize=9, fontweight="bold",
            color=col, transform=ax.transAxes)
    ax.text(0.06, 0.18, body, fontsize=8.5, color=C["grey_dark"],
            transform=ax.transAxes, wrap=True,
            multialignment="left", verticalalignment="bottom",
            linespacing=1.5)

fig.text(0.99, 0.01,
         "Qatar Airways | Meal Service A/B Test | PDD Portfolio | n=6,000",
         ha="right", va="bottom", fontsize=8, color=C["grey_dark"], alpha=0.5)

EXEC_PATH = "charts/fig_ab_05_executive_summary.png"
fig.savefig(EXEC_PATH, dpi=180, bbox_inches="tight", facecolor=C["off_white"])
plt.close(fig)
print(f"  ✓ Saved {EXEC_PATH}")



FIG AB-05 — Executive summary...
  ✓ Saved charts/fig_ab_05_executive_summary.png


In [15]:

# ── FINAL SUMMARY ─────────────────────────────────────────────────────────────
print()
print("=" * 65)
print("A/B TEST ANALYSIS COMPLETE")
print("=" * 65)
print(f"""
EXPERIMENT VERDICT: TREATMENT WINS — ROLL OUT RECOMMENDED

PRIMARY METRIC
  Meal satisfaction lift : +{lift_abs:.3f} pts  ({lift_pct:.1f}% improvement)
  Two-sample t-test      : t={t_stat:.3f},  p={t_p:.6f}  {sig_stars(t_p)}
  Mann-Whitney (non-par) : p={u_p:.6f}  {sig_stars(u_p)}
  Effect size            : Cohen's d = {d:.3f}  ({effect_label(d)})
  95% CI on lift         : [{treat_ci[0]-ctrl_ci[1]:.3f},  {treat_ci[1]-ctrl_ci[0]:.3f}]

SECONDARY METRICS
  Complaint rate         : {ctrl_complaint_rate:.1f}% → {treat_complaint_rate:.1f}%  (−{complaint_reduction:.1f}pp, p={p_c:.4f} {sig_stars(p_c)})
  Meal finish rate       : {ctrl['meal_finish_rate'].mean()*100:.1f}% → {treat['meal_finish_rate'].mean()*100:.1f}%  (p={p_fr:.4f} {sig_stars(p_fr)})
  Overall sat. impact    : p={p_oi:.4f}  {sig_stars(p_oi)}

SUBGROUP HIGHLIGHTS
  Economy lift           : +{cabin_df[cabin_df['cabin']=='Economy']['lift'].values[0]:.3f} pts  {sig_stars(cabin_df[cabin_df['cabin']=='Economy']['p'].values[0])}  ← highest ROI
  Business lift          : +{cabin_df[cabin_df['cabin']=='Business']['lift'].values[0]:.3f} pts  {sig_stars(cabin_df[cabin_df['cabin']=='Business']['p'].values[0])}
  First class lift       : +{cabin_df[cabin_df['cabin']=='First']['lift'].values[0]:.3f} pts  {sig_stars(cabin_df[cabin_df['cabin']=='First']['p'].values[0])}  ← at satisfaction ceiling

OUTPUTS
  fig_ab_01_overview.png         ← experiment design & balance checks
  fig_ab_02_statistical_tests.png ← full significance testing
  fig_ab_03_subgroup_analysis.png ← cabin, route, loyalty breakdowns
  fig_ab_04_business_case.png    ← financial projection & decision framework
  fig_ab_05_executive_summary.png ← boardroom-ready summary
""")



A/B TEST ANALYSIS COMPLETE

EXPERIMENT VERDICT: TREATMENT WINS — ROLL OUT RECOMMENDED

PRIMARY METRIC
  Meal satisfaction lift : +0.299 pts  (8.3% improvement)
  Two-sample t-test      : t=23.547,  p=0.000000  ***
  Mann-Whitney (non-par) : p=0.000000  ***
  Effect size            : Cohen's d = 0.609  (Medium)
  95% CI on lift         : [0.264,  0.334]

SECONDARY METRICS
  Complaint rate         : 8.0% → 4.3%  (−3.7pp, p=0.0000 ***)
  Meal finish rate       : 77.7% → 87.6%  (p=0.0000 ***)
  Overall sat. impact    : p=0.0000  ***

SUBGROUP HIGHLIGHTS
  Economy lift           : +0.341 pts  ***  ← highest ROI
  Business lift          : +0.215 pts  ***
  First class lift       : +0.076 pts  *  ← at satisfaction ceiling

OUTPUTS
  fig_ab_01_overview.png         ← experiment design & balance checks
  fig_ab_02_statistical_tests.png ← full significance testing
  fig_ab_03_subgroup_analysis.png ← cabin, route, loyalty breakdowns
  fig_ab_04_business_case.png    ← financial projection & decisi